# COMP5329 — Deep Learning

**Tutorial — Deep Generative Models**

**Semester 1, 2026**

### Learning Objectives
By the end of this tutorial you will be able to:
1. Describe the generative-modelling problem as density estimation + sampling, and place models into the **likelihood-based / implicit-adversarial / score-based** taxonomy.
2. Write the **GAN minimax objective** $\min_G\max_D V(D,G)$ and explain why the **non-saturating** generator loss fixes the vanishing-gradient failure mode of the original formulation.
3. **Implement** a minimal GAN from scratch — a Generator MLP, a Discriminator MLP, and a single training step — and train it on a 2-D toy distribution.
4. Write the **DDPM** forward noising equation $x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\varepsilon$, explain why we predict $\varepsilon$ (rather than $x_0$ or $\mu_\theta$), and state the reverse-step mean formula.
5. **Implement** DDPM's forward process, $\varepsilon$-prediction training loss, and one reverse step; train a small time-conditioned MLP and visualise sampling trajectories.
6. Answer exam-style questions on (i) the non-saturating generator loss, (ii) $\varepsilon$-prediction as score matching and why $\sim$1000 reverse steps are typical, and (iii) the high-level trade-off between adversarial and denoising generators.

### Topic Coverage

Week 12 covers **Deep Generative Models**. The full topic list (see `Week12_Self_Study_Deep_Generative_Models.ipynb`) is:

- ✅ **Taxonomy & problem setup** — density estimation vs sampling; likelihood-based vs implicit vs score-based *(tutorial)*
- ✅ **GANs (implicit / adversarial)** — minimax game, non-saturating loss, mode collapse, end-to-end implementation on 2-D toy data *(tutorial)*
- ✅ **DDPMs (denoising diffusion)** — forward noising, $\varepsilon$-prediction training, reverse-step update, end-to-end implementation on 2-D toy data *(tutorial)*
- 📖 **VAEs** — ELBO, reparameterisation trick, closed-form KL *(self-study — brief mention in Part A)*
- 📖 **DDIM** — non-Markovian deterministic fast sampler built on the same trained $\varepsilon_\theta$ *(self-study)*
- 📖 **Flow Matching & Rectified Flow** — straight-line transport between prior and data *(self-study — brief mention in Part A)*
- 📖 **Consistency Models** — single-step generation via self-consistency on a PF-ODE trajectory *(self-study — brief mention in Part A)*
- 📖 **WGAN / Wasserstein critics** — Lipschitz critic and weight clipping / gradient penalty *(self-study — one line in Part A)*

Due to time constraints the tutorial focuses on **two conceptually distinct** generators: **GANs** (implicit, adversarial) and **DDPMs** (denoising, score-based). VAEs, Flow Matching, Consistency Models, DDIM and WGAN are reviewed briefly in Part A and covered in depth in the self-study notebook.

The live session is organised into three parts: **Part A** — tutor walkthrough, **Part B** — in-class coding exercise, **Part C** — exam-style Q&A.

---
# Part A · Tutor Review

> **Goal.** By the end of Part A you should be able to (a) state the generative-modelling problem in one sentence, (b) place GAN and DDPM in the implicit-vs-score-based taxonomy, and (c) recite the two central formulas — the GAN minimax game and the DDPM forward equation — that Part B will turn into code.

---
## §0 — The Generative-Modelling Problem

Given i.i.d. samples $x_1,\dots,x_N\sim p_{\text{data}}(x)$ from an **unknown** distribution, a generative model tries to do one (or both) of:

1. **Density estimation** — learn $p_\theta(x)\approx p_{\text{data}}(x)$ that can be *evaluated* at any point $x$.
2. **Sampling** — produce new samples $\tilde x\sim p_\theta$ that *look like* $p_{\text{data}}$.

On high-dimensional data (images, audio, molecules) the density is intractable. So modern deep generative models give up on one of the two goals and commit to a **trade-off**:

| Family | Likelihood $p_\theta(x)$? | Sampling? | Example |
|---|---|---|---|
| **Likelihood-based (explicit)** | Yes — ELBO or exact | Indirect (via decoder / change-of-variables) | VAE, normalising flows, autoregressive models |
| **Implicit (adversarial)** | **No** — only a sampler | Direct (one forward pass through $G$) | **GAN** |
| **Score-based / diffusion** | Implicit (via the score) | Iterative (denoising chain) | **DDPM**, score-SDE, Flow Matching |

Both of the models we implement today sidestep the intractable likelihood of $p_\theta$, but for **opposite** reasons:
- **GANs** never write down a density at all — they learn a **sampler** $G(z)$ and a **critic** $D$ that tells them when the sampler is wrong.
- **DDPMs** *do* model a density, but only *implicitly* through the **score** $\nabla_x\log p_t(x)$ of a sequence of noise-corrupted versions of the data — they never evaluate $p_\theta(x)$ directly either.

---
## §1 — GANs: the Adversarial Viewpoint

### The minimax game
A GAN pits two networks against each other:
- A **generator** $G_\theta:\mathbb{R}^{d_z}\to\mathbb{R}^d$ that maps noise $z\sim p_z$ (usually $\mathcal{N}(0,I)$) to samples $G(z)$.
- A **discriminator** $D_\phi:\mathbb{R}^d\to(0,1)$ that outputs the probability that its input is **real**.

The training objective of Goodfellow et al. (2014) is the two-player zero-sum game
$$\min_{G}\;\max_{D}\;V(D,G)\;=\;\mathbb{E}_{x\sim p_{\text{data}}}[\log D(x)]\;+\;\mathbb{E}_{z\sim p_z}[\log(1-D(G(z)))].$$

- The **discriminator** maximises $V$: it wants $D(x)\to 1$ on real data and $D(G(z))\to 0$ on fakes. This is just **binary cross-entropy** with real=1, fake=0.
- The **generator** minimises $V$: holding $D$ fixed, the only term that depends on $\theta$ is $\mathbb{E}_z[\log(1-D(G(z)))]$, so $G$ wants $D(G(z))\to 1$.

At the global optimum (with infinite capacity) $D^\star(x)=\tfrac{p_{\text{data}}(x)}{p_{\text{data}}(x)+p_G(x)}=\tfrac12$ and $p_G=p_{\text{data}}$ — the generator has matched the data distribution and the discriminator cannot tell real from fake.

### The non-saturating generator loss (what we actually use)
The theoretical generator loss $\log(1-D(G(z)))$ is a **disaster** early in training. When $G$ is bad, $D$ easily classifies fakes as fake, so $D(G(z))\approx 0$ and $\log(1-D(G(z)))\approx\log 1=0$. The gradient $\partial_\theta\log(1-D(G(z)))$ then **vanishes** exactly when we need it most — the generator has no signal to improve.

Goodfellow's fix: replace the minimisation of $\log(1-D(G(z)))$ with the **maximisation** of
$$\mathcal{L}_G^{\text{NS}}(\theta)\;=\;-\,\mathbb{E}_{z}[\log D(G(z))].$$
This is the **non-saturating** form. It has the same fixed point ($D(G(z))=\tfrac12$) but a much healthier gradient when $D(G(z))$ is near 0 — because $-\log D$ **blows up** there rather than flattening out. In code this is implemented as a BCE with target = 1 on the fakes (i.e. "train the generator to *fool* the discriminator into saying real").

### Mode collapse — the central failure mode
Because the generator is only asked to produce samples $D$ cannot distinguish, it has no incentive to cover the whole data distribution. A generator that finds **one** very convincing output (a single mode) and emits it repeatedly can score arbitrarily well until $D$ adapts. The result is **mode collapse**: $G$ keeps producing near-identical samples and ignores entire regions of $p_{\text{data}}$. Symptoms in our 2-D demo: the generator fits one Gaussian out of eight and never discovers the others.

> **Wasserstein GANs** (WGAN, Arjovsky et al. 2017) replace the JS-divergence game with a **Wasserstein-1** distance estimated by a **1-Lipschitz critic** (enforced via weight clipping or a gradient penalty). This smooths the loss landscape and reduces mode collapse, but the core adversarial structure is identical — we do not implement WGAN here; see the self-study notebook.

---
## §2 — DDPMs: the Denoising / Score-Based Viewpoint

### Forward process — gradually destroy the signal
Fix a variance schedule $\beta_1,\dots,\beta_T\in(0,1)$ (small, increasing). The **forward** process turns a clean sample $x_0\sim p_{\text{data}}$ into pure noise over $T$ steps by iterating
$$q(x_t\mid x_{t-1})\;=\;\mathcal{N}\!\big(\sqrt{1-\beta_t}\,x_{t-1},\,\beta_t I\big).$$
Because every step is Gaussian, we can **marginalise out all the intermediate steps** and sample $x_t$ from $x_0$ in a **single** closed-form draw. Let $\alpha_t:=1-\beta_t$ and $\bar\alpha_t:=\prod_{s\le t}\alpha_s$; then
$$\boxed{\;x_t\;=\;\sqrt{\bar\alpha_t}\,x_0\;+\;\sqrt{1-\bar\alpha_t}\,\varepsilon,\qquad \varepsilon\sim\mathcal{N}(0,I)\;}$$
At $t=0$: $\bar\alpha_0\approx 1$ and $x_0=x_0$. At $t=T$: $\bar\alpha_T\approx 0$ and $x_T\approx \varepsilon\sim\mathcal{N}(0,I)$ — the signal is gone.

### Training objective — predict the noise
The reverse kernel $q(x_{t-1}\mid x_t,x_0)$ is a tractable Gaussian whose mean depends on $x_0$. We parameterise a network to predict the quantity that determines this mean. Ho et al. (2020) showed that, up to re-weighting, the variational lower bound reduces to the simple denoising objective
$$\mathcal{L}_{\text{simple}}(\theta)\;=\;\mathbb{E}_{x_0,\,t\sim\mathcal{U}\{1,\dots,T\},\,\varepsilon\sim\mathcal{N}(0,I)}\!\left[\,\big\lVert\varepsilon-\varepsilon_\theta(x_t,t)\big\rVert^2\right].$$
That is: sample a clean data point $x_0$, pick a random timestep $t$, draw a random noise $\varepsilon$, forward-noise to $x_t$, and train the network to **recover the exact noise that was added**. It is just an MSE.

**Why predict $\varepsilon$ and not $x_0$ or $\mu_\theta$?** The target $\varepsilon\sim\mathcal{N}(0,I)$ has **timestep-independent unit scale** — the loss is comparable across all $t$ and trains with a single learning rate. Targets $x_0$ (data scale) and $\mu_\theta$ (scaled by $\sqrt{\alpha_t}$) vary drastically with $t$: near $t=T$ the signal is buried in noise, the target is tiny, and the network learns nothing at high $t$. $\varepsilon$-prediction *is* denoising score matching — see Part C, Q2.

### Reverse step — one denoising update
Given $x_t$ and the predicted noise $\varepsilon_\theta(x_t,t)$, the reverse step computes the posterior mean and adds a stochastic kick:
$$\mu_\theta(x_t,t)\;=\;\frac{1}{\sqrt{\alpha_t}}\!\left(x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\varepsilon_\theta(x_t,t)\right),\qquad x_{t-1}=\mu_\theta(x_t,t)+\sigma_t z,\;\;z\sim\mathcal{N}(0,I),$$
with $\sigma_t^2=\beta_t$ (a standard choice) and $z=0$ at $t=1$. Iterating this from $x_T\sim\mathcal{N}(0,I)$ down to $x_0$ is the DDPM sampler. Typical $T=1000$ on images; for our 2-D toy problem $T=200$ is plenty.

---
## §3 — A quick look at the forward process

Before Part B, let's visualise the DDPM forward process on 2-D data. This is the *only* code in Part A — it is purely illustrative. The small helper below also pre-computes the $\bar\alpha_t$ schedule that Part B will use.

In [ ]:
# ── Part A illustrative plot: forward noising of 2-D data ──────────────────
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

%matplotlib inline
torch.manual_seed(0)

# Toy 2-D target: 8 Gaussians in a ring (shared by both GAN and DDPM demos)
def make_8gaussians(n=2000, std=0.05, radius=2.0):
    angles = torch.linspace(0, 2 * math.pi, 9)[:-1]
    centres = torch.stack([torch.cos(angles), torch.sin(angles)], 1) * radius
    idx = torch.randint(0, 8, (n,))
    return centres[idx] + torch.randn(n, 2) * std

data_2d = make_8gaussians(2000)

# Linear beta schedule -> alpha_bar
T = 200
betas = torch.linspace(1e-4, 0.02, T)
alpha_bars = torch.cumprod(1.0 - betas, dim=0)

# Show x_t at a few timesteps
snapshots = [0, 20, 60, 120, 199]
fig, axes = plt.subplots(1, len(snapshots), figsize=(14, 2.8))
for ax, t in zip(axes, snapshots):
    ab = alpha_bars[t]
    xt = torch.sqrt(ab) * data_2d + torch.sqrt(1 - ab) * torch.randn_like(data_2d)
    ax.scatter(xt[:, 0], xt[:, 1], s=2, alpha=0.4)
    ax.set_title(f"t = {t}")
    ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_aspect('equal')
plt.suptitle("Forward process: clean data (t=0) → Gaussian noise (t=T)")
plt.tight_layout(); plt.show()

---
## §3.5 — Very brief notes on models we are *not* implementing

These are covered in depth in the self-study notebook; we list them here so the taxonomy is complete.

- **VAE.** Trains an encoder $q_\phi(z\mid x)$ and decoder $p_\theta(x\mid z)$ to maximise the ELBO $=\mathbb{E}_{q_\phi}[\log p_\theta(x\mid z)]-\mathrm{KL}(q_\phi\Vert p)$ — a **reconstruction** term minus a **KL regulariser**. The **reparameterisation trick** $z=\mu_\phi(x)+\sigma_\phi(x)\odot\epsilon$ lets gradients flow through the sampling step. VAEs share DDPM's ELBO/score-matching DNA (DDPM can be derived as a hierarchical VAE), which is exactly why the instructor paired GAN with DDPM instead of VAE with DDPM.

- **Flow Matching.** Trains a velocity field $v_\theta(x,t)$ to match a target vector field that transports the prior $\mathcal{N}(0,I)$ to $p_{\text{data}}$ along (ideally) **straight-line** paths $x_t=(1-t)x_0+t\,x_1$. Sampling is Euler integration of an ODE — far fewer steps than DDPM, same trained-on-data-pairs regime. **Rectified Flow** iterates this to make the paths progressively straighter.

- **Consistency Models.** Distil a trained diffusion model into a map $f_\theta(x_t,t)$ that is **self-consistent** along a single PF-ODE trajectory — i.e. $f_\theta$ sends every point on the trajectory directly to its $t=0$ endpoint. This gives **one-step** (or few-step) sampling with diffusion-level quality.

- **WGAN.** Replaces the JS-divergence game with a Wasserstein-1 distance estimated by a 1-Lipschitz critic (weight clipping or gradient penalty) — smoother loss landscape, fewer mode-collapse failures.

---
# Part B · In-Class Exercise

You will implement **two** generative models end-to-end on the same 2-D toy distribution:

1. **Task 1 — GAN**: Generator MLP, Discriminator MLP, and a single non-saturating training step.
2. **Task 2 — DDPM**: forward `q_sample`, $\varepsilon$-prediction loss, and one reverse step.

Both tasks share the `data_2d`, `betas`, and `alpha_bars` you computed in Part A.

---
## Task 1 · GAN on a 2-D Ring of Gaussians

You will build a GAN that maps a 16-dimensional latent $z\sim\mathcal{N}(0,I)$ to 2-D points matching `data_2d`. There are **three** `# TODO`s:
1. Finish `Generator.forward` (a tiny 3-layer MLP).
2. Finish `Discriminator.forward` (a tiny 3-layer MLP producing a single logit).
3. Finish `gan_step`: one discriminator update (max $\log D(x)+\log(1-D(G(z)))$) **then** one generator update using the **non-saturating** loss $-\log D(G(z))$.

All three are plain `F.binary_cross_entropy_with_logits` calls once you remember that:
- BCE-with-logits with target **1** on reals implements $-\log D(x)$.
- BCE-with-logits with target **0** on fakes implements $-\log(1-D(G(z)))$.
- The non-saturating generator loss is BCE-with-logits with target **1** on fakes — i.e. "train $G$ to convince $D$ that fakes are real".

### Task B1 · Fill in `Generator` and `Discriminator`

In [ ]:
LATENT_DIM = 16

class Generator(nn.Module):
    '''z (B, LATENT_DIM) -> x (B, 2).  A tiny MLP.'''
    def __init__(self, latent_dim=LATENT_DIM, hidden=64, out_dim=2):
        super().__init__()
        self.fc1 = nn.Linear(latent_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, out_dim)

    def forward(self, z):
        # TODO 1 — two hidden ReLU layers, then a linear output (no activation)
        h = ...
        h = ...
        x = ...
        return x


class Discriminator(nn.Module):
    '''x (B, 2) -> logit (B, 1).  Outputs a LOGIT (no sigmoid).'''
    def __init__(self, in_dim=2, hidden=64):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, 1)

    def forward(self, x):
        # TODO 2 — two hidden LeakyReLU(0.2) layers, then a linear output (logit)
        h = ...
        h = ...
        logit = ...
        return logit

<details>
<summary><b>▸ Solution · Task B1</b> (click to expand)</summary>

```python
class Generator(nn.Module):
    def forward(self, z):
        h = F.relu(self.fc1(z))         # TODO 1a
        h = F.relu(self.fc2(h))         # TODO 1b
        x = self.fc3(h)                 # TODO 1c — linear output
        return x

class Discriminator(nn.Module):
    def forward(self, x):
        h = F.leaky_relu(self.fc1(x), 0.2)  # TODO 2a
        h = F.leaky_relu(self.fc2(h), 0.2)  # TODO 2b
        logit = self.fc3(h)                 # TODO 2c — raw logit, no sigmoid
        return logit
```

**Key points:**
- **Output layers are *linear*.** The generator outputs points in $\mathbb{R}^2$ — no activation would restrict the range. The discriminator outputs a **logit**; the sigmoid is absorbed into `binary_cross_entropy_with_logits` for numerical stability (this is the standard GAN recipe).
- **LeakyReLU on $D$.** If $D$ uses plain ReLU and a neuron dies on a real sample, its gradient is 0 forever and $G$ receives no signal through that channel. LeakyReLU's small negative slope keeps every neuron alive — a small but important trick in GAN training.
- **Capacity matters.** A 64-hidden-unit MLP is enough for 8 Gaussians in 2-D. Too small → under-fitting (generator can only cover a few modes); too large relative to the data → $D$ overpowers $G$ and training collapses.
</details>

### Task B2 · Fill in `gan_step` (one training step)

`gan_step` takes a real minibatch and performs **two** optimiser updates: one for $D$, then one for $G$. Remember to `.detach()` the fake batch during the $D$ step so the generator's parameters are **not** updated by the discriminator loss. Use the **non-saturating** generator loss (target = 1 on fakes).

In [ ]:
def gan_step(G, D, real_batch, g_opt, d_opt, latent_dim=LATENT_DIM):
    '''
    One GAN training step (non-saturating).
      real_batch : (B, 2) tensor of real samples
      Returns (d_loss, g_loss) as floats.
    '''
    B = real_batch.size(0)
    device = real_batch.device
    real_labels = torch.ones (B, 1, device=device)
    fake_labels = torch.zeros(B, 1, device=device)

    # ── (A) Discriminator step: maximise log D(x) + log(1 - D(G(z))) ──────
    z = torch.randn(B, latent_dim, device=device)
    fake = G(z).detach()                                          # stop grads to G

    d_real_logit = D(real_batch)
    d_fake_logit = D(fake)
    # TODO 3a — BCE-with-logits: real→1, fake→0, sum the two halves
    d_loss_real = ...
    d_loss_fake = ...
    d_loss = d_loss_real + d_loss_fake

    d_opt.zero_grad()
    d_loss.backward()
    d_opt.step()

    # ── (B) Generator step: NON-SATURATING loss  −log D(G(z)) ─────────────
    z = torch.randn(B, latent_dim, device=device)
    fake = G(z)                                                    # NO detach here
    d_fake_logit_for_g = D(fake)
    # TODO 3b — non-saturating G loss: BCE-with-logits with target = 1 on fakes
    g_loss = ...

    g_opt.zero_grad()
    g_loss.backward()
    g_opt.step()

    return d_loss.item(), g_loss.item()

<details>
<summary><b>▸ Solution · Task B2</b> (click to expand)</summary>

```python
def gan_step(G, D, real_batch, g_opt, d_opt, latent_dim=LATENT_DIM):
    B = real_batch.size(0); device = real_batch.device
    real_labels = torch.ones (B, 1, device=device)
    fake_labels = torch.zeros(B, 1, device=device)

    # (A) Discriminator
    z = torch.randn(B, latent_dim, device=device)
    fake = G(z).detach()
    d_real_logit = D(real_batch)
    d_fake_logit = D(fake)
    d_loss_real = F.binary_cross_entropy_with_logits(d_real_logit, real_labels)  # TODO 3a
    d_loss_fake = F.binary_cross_entropy_with_logits(d_fake_logit, fake_labels)  # TODO 3a
    d_loss = d_loss_real + d_loss_fake
    d_opt.zero_grad(); d_loss.backward(); d_opt.step()

    # (B) Generator — NON-SATURATING
    z = torch.randn(B, latent_dim, device=device)
    fake = G(z)
    d_fake_logit_for_g = D(fake)
    g_loss = F.binary_cross_entropy_with_logits(d_fake_logit_for_g, real_labels) # TODO 3b
    g_opt.zero_grad(); g_loss.backward(); g_opt.step()
    return d_loss.item(), g_loss.item()
```

**Key points:**
- **`.detach()` during the D step** is essential. Without it, `d_loss.backward()` would also populate `G.parameters().grad`, and `g_opt.step()` would move $G$ in the wrong direction using a loss it was never meant to see.
- **Two fresh `z` draws.** The generator step re-samples $z$ so that the $G$ update uses a different minibatch of fakes from the $D$ update. In practice either convention works, but re-sampling slightly decorrelates the two losses.
- **Non-saturating = BCE with target 1 on fakes.** Algebraically, $-\log D(G(z))$ is the BCE of the fake logits against the label "real". This is the single-line change from the original formulation $\log(1-D(G(z)))$ — and the reason training does not stall when $D$ is strong.
- **BCE-with-logits** is numerically safer than `log(sigmoid(...))` (no over/underflow).
</details>

### Demo · Train the GAN for 500 steps on the 2-D ring

In [ ]:
torch.manual_seed(0)
G = Generator(); D = Discriminator()
g_opt = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
d_opt = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

BATCH = 256
for step in range(500):
    idx = torch.randint(0, len(data_2d), (BATCH,))
    real = data_2d[idx]
    d_l, g_l = gan_step(G, D, real, g_opt, d_opt)
    if (step + 1) % 100 == 0:
        print(f"step {step+1:4d}   D_loss={d_l:.3f}   G_loss={g_l:.3f}")

# Sample from the trained generator
with torch.no_grad():
    z = torch.randn(2000, LATENT_DIM)
    fake = G(z)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].scatter(data_2d[:, 0], data_2d[:, 1], s=2, alpha=0.3, c='C0')
axes[0].set_title("Real (8 Gaussians)")
axes[1].scatter(fake[:, 0], fake[:, 1], s=2, alpha=0.3, c='C3')
axes[1].set_title("GAN samples (500 steps)")
for ax in axes:
    ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_aspect('equal')
plt.tight_layout(); plt.show()

> **What you should see.** After 500 steps the GAN typically covers *several* (not always all 8) of the modes. If you see it covering only one blob — that is **mode collapse** in action. Try increasing `BATCH`, re-running with a different seed, or giving $G$ slightly less capacity; this is a well-known GAN failure mode, not a bug in your code. Part C · Q3 revisits the coverage-vs-quality trade-off.

---
## Task 2 · DDPM on the same 2-D Ring

You will now build a DDPM on the same `data_2d`. There are **three** `# TODO`s:
1. Finish `q_sample` — the closed-form forward equation $x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\varepsilon$.
2. Finish `ddpm_loss` — sample $t$, sample $\varepsilon$, forward-noise, MSE between $\varepsilon$ and $\varepsilon_\theta(x_t,t)$.
3. Finish `ddpm_reverse_step` — one step of the reverse update $\mu_\theta+\sigma_t z$.

The noise predictor is given — a tiny MLP with a sinusoidal time embedding (conceptually identical to the positional encoding from Week 7).

In [ ]:
# ── Sinusoidal time embedding + noise-predictor MLP (PROVIDED) ─────────────

class SinusoidalTimeEmb(nn.Module):
    def __init__(self, dim): super().__init__(); self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        args = t.unsqueeze(-1).float() * freqs.unsqueeze(0)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

class NoisePredictor(nn.Module):
    '''MLP that predicts noise given (x_t, t).  x_t: (B,2), t: (B,)'''
    def __init__(self, data_dim=2, hidden=128, time_dim=32):
        super().__init__()
        self.time_emb = SinusoidalTimeEmb(time_dim)
        self.net = nn.Sequential(
            nn.Linear(data_dim + time_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden),              nn.SiLU(),
            nn.Linear(hidden, data_dim))
    def forward(self, x, t):
        return self.net(torch.cat([x, self.time_emb(t)], dim=-1))

### Task B3 · Fill in `q_sample` (forward process)

In [ ]:
def q_sample(x0, t, noise, sqrt_alpha_bar, sqrt_one_minus_alpha_bar):
    '''
    Closed-form forward noising:  x_t = sqrt(alpha_bar_t) * x0 + sqrt(1 - alpha_bar_t) * noise
      x0    : (B, 2)
      t     : (B,)  integer timesteps in [0, T)
      noise : (B, 2)  drawn from N(0, I)
      sqrt_alpha_bar, sqrt_one_minus_alpha_bar : (T,) precomputed schedules
    Returns x_t : (B, 2).
    '''
    # TODO 4 — index the schedules at t, reshape to (B, 1), apply the forward formula
    sa  = ...    # (B, 1)
    sma = ...    # (B, 1)
    x_t = ...
    return x_t

<details>
<summary><b>▸ Solution · Task B3</b> (click to expand)</summary>

```python
def q_sample(x0, t, noise, sqrt_alpha_bar, sqrt_one_minus_alpha_bar):
    sa  = sqrt_alpha_bar[t].unsqueeze(-1)             # TODO 4a
    sma = sqrt_one_minus_alpha_bar[t].unsqueeze(-1)   # TODO 4b
    x_t = sa * x0 + sma * noise                       # TODO 4c
    return x_t
```

**Key points:**
- **Closed-form marginal.** Although the forward process is defined step-by-step, its composition is Gaussian and we get $x_t$ in **one** draw. This is the whole reason training is tractable: we never have to simulate the $T$-step chain during training.
- **`unsqueeze(-1)` is mandatory.** `sqrt_alpha_bar` has shape `(T,)`; after gather we get `(B,)`; we need `(B, 1)` so it broadcasts against `x0` of shape `(B, 2)`. Forgetting this silently gives a `(B, B)` outer product and you will chase ghosts for an hour.
- **Decoupled from the network.** `q_sample` involves **no learned parameters** — the forward process is fixed. Only the reverse process is learned (via `ddpm_loss` below).
</details>

### Task B4 · Fill in `ddpm_loss` (training objective)

In [ ]:
def ddpm_loss(eps_model, x0, t_batch, schedule):
    '''
    Simple DDPM loss  E[|| eps - eps_theta(x_t, t) ||^2].
      eps_model : network (x_t, t) -> predicted noise
      x0        : (B, 2) clean samples
      t_batch   : (B,)  integer timesteps drawn uniformly in [0, T)
      schedule  : dict with sqrt_alpha_bar, sqrt_one_minus_alpha_bar  (both shape (T,))
    '''
    sab  = schedule['sqrt_alpha_bar']
    smab = schedule['sqrt_one_minus_alpha_bar']

    # TODO 5a — draw noise with the same shape as x0
    noise = ...

    # TODO 5b — forward-noise x0 to x_t using q_sample
    x_t = ...

    # TODO 5c — predict noise and compute the MSE loss
    eps_pred = eps_model(x_t, t_batch.float())
    loss = ...
    return loss

<details>
<summary><b>▸ Solution · Task B4</b> (click to expand)</summary>

```python
def ddpm_loss(eps_model, x0, t_batch, schedule):
    sab  = schedule['sqrt_alpha_bar']
    smab = schedule['sqrt_one_minus_alpha_bar']
    noise = torch.randn_like(x0)                              # TODO 5a
    x_t   = q_sample(x0, t_batch, noise, sab, smab)           # TODO 5b
    eps_pred = eps_model(x_t, t_batch.float())
    loss  = F.mse_loss(eps_pred, noise)                       # TODO 5c
    return loss
```

**Key points:**
- **Uniform $t$ sampling.** Every timestep is trained with equal weight by averaging over `t_batch ~ Uniform{0,...,T-1}`. The *weighted* ELBO has $t$-dependent coefficients; Ho et al. (2020) show that the **un-weighted** MSE is equivalent (up to a reweighting) and works better in practice — that's exactly what we use here.
- **MSE against $\varepsilon$.** The target is a standard normal, so the loss has a **constant scale across $t$** — the main reason $\varepsilon$-prediction out-performs predicting $x_0$ or $\mu_\theta$. Part C · Q2 proves this is denoising score matching in disguise.
- **No autograd surprises.** `x_t` depends on `x0` and `noise`, neither of which carries gradients through the network. Only `eps_pred` back-propagates.
</details>

### Task B5 · Fill in `ddpm_reverse_step` (one denoising step)

In [ ]:
@torch.no_grad()
def ddpm_reverse_step(eps_model, x_t, t, schedule):
    '''
    One reverse-process step: x_t -> x_{t-1}.
      x_t : (B, 2)
      t   : int (the current timestep)
      schedule : dict with betas, alphas, alpha_bars  (each shape (T,))
    Returns x_{t-1} : (B, 2).
    '''
    betas      = schedule['betas']
    alphas     = schedule['alphas']
    alpha_bars = schedule['alpha_bars']

    B = x_t.size(0)
    t_batch = torch.full((B,), t, dtype=torch.float, device=x_t.device)
    eps_pred = eps_model(x_t, t_batch)

    beta_t = betas[t]
    alpha_t = alphas[t]
    alpha_bar_t = alpha_bars[t]

    # TODO 6a — posterior mean:
    #   mu = (1/sqrt(alpha_t)) * ( x_t - beta_t / sqrt(1 - alpha_bar_t) * eps_pred )
    mu = ...

    # TODO 6b — stochastic kick with sigma_t = sqrt(beta_t) for t > 0, else zero
    if t > 0:
        z = torch.randn_like(x_t)
        sigma_t = torch.sqrt(beta_t)
        x_prev = ...
    else:
        x_prev = mu
    return x_prev

<details>
<summary><b>▸ Solution · Task B5</b> (click to expand)</summary>

```python
@torch.no_grad()
def ddpm_reverse_step(eps_model, x_t, t, schedule):
    betas, alphas, alpha_bars = schedule['betas'], schedule['alphas'], schedule['alpha_bars']
    B = x_t.size(0)
    t_batch = torch.full((B,), t, dtype=torch.float, device=x_t.device)
    eps_pred = eps_model(x_t, t_batch)

    beta_t, alpha_t, alpha_bar_t = betas[t], alphas[t], alpha_bars[t]
    # TODO 6a
    mu = (1.0 / torch.sqrt(alpha_t)) * (
             x_t - (beta_t / torch.sqrt(1.0 - alpha_bar_t)) * eps_pred)
    # TODO 6b
    if t > 0:
        z = torch.randn_like(x_t)
        sigma_t = torch.sqrt(beta_t)
        x_prev = mu + sigma_t * z
    else:
        x_prev = mu
    return x_prev
```

**Key points:**
- **Where the formula comes from.** It is the posterior mean of the tractable Gaussian $q(x_{t-1}\mid x_t,x_0)$ after substituting $\hat x_0=(x_t-\sqrt{1-\bar\alpha_t}\,\varepsilon_\theta)/\sqrt{\bar\alpha_t}$. That is, given the current $x_t$ and the network's best guess of the noise, we reconstruct a point estimate of $x_0$ and take one step back along the chain.
- **Two common choices of $\sigma_t$:** (i) $\sigma_t^2=\beta_t$ — upper bound on the true posterior variance, what we use here; (ii) $\sigma_t^2=\tilde\beta_t=\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\beta_t$ — the exact posterior variance. Both give comparable quality.
- **No kick at $t=0$.** At the final step we emit the mean directly — adding noise at $t=0$ would just re-corrupt the sample we worked hard to denoise.
- **`@torch.no_grad()`** is applied because sampling never needs gradients — we only differentiate through the training loss.
</details>

### Demo · Train for 1000 steps, then sample trajectories

In [ ]:
# ── Build the schedule dict Part B tasks refer to ──────────────────────────
T = 200
betas        = torch.linspace(1e-4, 0.02, T)
alphas       = 1.0 - betas
alpha_bars   = torch.cumprod(alphas, dim=0)
schedule = dict(
    betas=betas, alphas=alphas, alpha_bars=alpha_bars,
    sqrt_alpha_bar=torch.sqrt(alpha_bars),
    sqrt_one_minus_alpha_bar=torch.sqrt(1.0 - alpha_bars))

# ── Train the noise predictor ──────────────────────────────────────────────
torch.manual_seed(0)
eps_model = NoisePredictor()
opt = torch.optim.Adam(eps_model.parameters(), lr=2e-3)

for step in range(1000):
    idx = torch.randint(0, len(data_2d), (256,))
    x0 = data_2d[idx]
    t_batch = torch.randint(0, T, (256,))
    loss = ddpm_loss(eps_model, x0, t_batch, schedule)
    opt.zero_grad(); loss.backward(); opt.step()
    if (step + 1) % 200 == 0:
        print(f"step {step+1:4d}   loss={loss.item():.4f}")

# ── Sample 2000 points, saving a few trajectory snapshots ─────────────────
N = 2000
x = torch.randn(N, 2)
snapshots = {T - 1: x.clone()}
save_at = {0, T // 4, T // 2, 3 * T // 4, T - 1}
for t in reversed(range(T)):
    x = ddpm_reverse_step(eps_model, x, t, schedule)
    if t in save_at:
        snapshots[t] = x.clone()

fig, axes = plt.subplots(1, len(save_at), figsize=(14, 2.8))
for ax, t in zip(axes, sorted(save_at, reverse=True)):
    ax.scatter(snapshots[t][:, 0], snapshots[t][:, 1], s=2, alpha=0.4)
    ax.set_title(f"t = {t}")
    ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_aspect('equal')
plt.suptitle("DDPM reverse trajectory: noise (t=T-1) → data (t=0)")
plt.tight_layout(); plt.show()

> **What you should see.** The first panel is roughly $\mathcal{N}(0,I)$; successive panels concentrate on the 8 Gaussians; by $t=0$ the samples tile all eight modes. Unlike the GAN, a working DDPM **rarely mode-collapses** — the forward process guarantees every mode has nonzero measure in the training distribution, and the MSE loss rewards covering all of them.

---
# Part C · Exam-Style Questions

Three questions, each with three sub-parts. Try to answer before expanding the model solution.

## Q1 · Why the non-saturating GAN loss?  *(7 marks)*

The original GAN paper's generator minimises $\log(1-D(G(z)))$ but the implementation we wrote in Task B2 maximises $\log D(G(z))$ (equivalently, BCE with target = 1 on fakes).

**(a)** *(3 marks)* Consider the **saturating** loss $\mathcal{L}_G^{\text{sat}}(\theta)=\mathbb{E}_z[\log(1-D(G(z)))]$. Compute $\partial \mathcal{L}_G^{\text{sat}}/\partial D(G(z))$ and show that it tends to **0** as $D(G(z))\to 0$. Explain in one sentence why this is exactly the regime *early* training is in.

**(b)** *(2 marks)* Now consider the **non-saturating** loss $\mathcal{L}_G^{\text{NS}}(\theta)=-\mathbb{E}_z[\log D(G(z))]$. Compute $\partial\mathcal{L}_G^{\text{NS}}/\partial D(G(z))$ and describe its behaviour as $D(G(z))\to 0$. Contrast with part (a).

**(c)** *(2 marks)* The two losses share the **same fixed point** $D(G(z))=\tfrac12$ (at which $G$ stops moving). Given that, why do we still bother to switch? Answer in terms of what the gradient contributes when $D$ is *confidently wrong* about the generator's current output.

<details><summary><b>▸ Answer sketch — Q1</b></summary>

**(a)** With $u:=D(G(z))$ we have $\mathcal{L}_G^{\text{sat}}=\log(1-u)$, so
$$\frac{\partial \mathcal{L}_G^{\text{sat}}}{\partial u}=\frac{-1}{1-u}.$$
As $u\to 0$ this derivative tends to $-1$ in magnitude, but what matters is the **upstream** gradient that reaches $\theta$: by chain rule
$$\frac{\partial \mathcal{L}_G^{\text{sat}}}{\partial \theta}=\frac{\partial \mathcal{L}_G^{\text{sat}}}{\partial u}\,\frac{\partial u}{\partial \theta}=\frac{-1}{1-u}\,\sigma(z_{\text{logit}})(1-\sigma(z_{\text{logit}}))\frac{\partial z_{\text{logit}}}{\partial\theta}=\frac{-u(1-u)}{1-u}\,\frac{\partial z_{\text{logit}}}{\partial\theta}=-u\,\frac{\partial z_{\text{logit}}}{\partial\theta}.$$
As $u=D(G(z))\to 0$ the factor $u$ kills the gradient. **Early in training the generator is bad, $D$ confidently classifies fakes as fake ($u\approx 0$), and the generator receives essentially no gradient** — it cannot climb out of the hole.

**(b)** Now $\mathcal{L}_G^{\text{NS}}=-\log u$, so $\partial/\partial u=-1/u$ and the chain rule gives
$$\frac{\partial \mathcal{L}_G^{\text{NS}}}{\partial\theta}=-\frac{1}{u}\,u(1-u)\,\frac{\partial z_{\text{logit}}}{\partial\theta}=-(1-u)\,\frac{\partial z_{\text{logit}}}{\partial\theta}.$$
As $u\to 0$ the coefficient tends to $-1$: the generator still gets **strong** signal. As $u\to 1$ (generator is winning) the coefficient shrinks toward 0 — exactly the regime where we no longer need to push.

**(c)** Both losses are stationary when $u=\tfrac12$ (strictly, both have the same global optimum for $G$ against the optimal $D^\star$), but **the saturating loss has near-zero gradient whenever $D$ is confidently right and $G$ is losing**, which is *precisely* the early-training regime. The non-saturating loss delivers a gradient of order $1$ in that regime and only fades when training has already succeeded. This is a pure optimisation fix — same game, same fixed points, very different gradient fields — and it is why every modern GAN codebase uses it.
</details>

## Q2 · $\varepsilon$-prediction is score matching  *(7 marks)*

DDPM trains a network $\varepsilon_\theta(x_t,t)$ with
$$\mathcal{L}(\theta)=\mathbb{E}_{x_0,\varepsilon,t}\!\left[\,\lVert\varepsilon_\theta(x_t,t)-\varepsilon\rVert^2\right],\qquad x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\varepsilon.$$

**(a)** *(3 marks)* Using $q(x_t\mid x_0)=\mathcal{N}(\sqrt{\bar\alpha_t}\,x_0,(1-\bar\alpha_t)I)$, show that
$$\nabla_{x_t}\log q(x_t\mid x_0)\;=\;-\frac{x_t-\sqrt{\bar\alpha_t}\,x_0}{1-\bar\alpha_t}\;=\;-\frac{\varepsilon}{\sqrt{1-\bar\alpha_t}}.$$
Hence give the formula that converts a trained noise predictor into an estimate of the *marginal* score $s_\theta(x_t,t)\approx\nabla_{x_t}\log p_t(x_t)$.

**(b)** *(2 marks)* DDPM could equivalently be parameterised to predict $x_0$ directly, or the posterior mean $\mu_\theta$. Why is **$\varepsilon$-prediction** preferred in practice? Answer in terms of the *scale* of the target across timesteps and what that implies for the loss landscape near $t\to 0$ and $t\to T$.

**(c)** *(2 marks)* On high-resolution images, DDPM typically uses $T\approx 1000$ reverse steps. Explain **why 1000** — i.e. what goes wrong if you train with $T=1000$ but then run the reverse chain with only, say, 10 steps of the *DDPM* updater (not a fancy DDIM sampler)? Phrase your answer in terms of the local-Markov assumption used to derive the DDPM update formula in Task B5.

<details><summary><b>▸ Answer sketch — Q2</b></summary>

**(a)** For $q(x_t\mid x_0)=\mathcal{N}(\mu,\sigma^2 I)$ with $\mu=\sqrt{\bar\alpha_t}\,x_0$ and $\sigma^2=1-\bar\alpha_t$,
$$\log q(x_t\mid x_0)=-\frac{\lVert x_t-\mu\rVert^2}{2\sigma^2}+\text{const}\;\Longrightarrow\;\nabla_{x_t}\log q=-\frac{x_t-\sqrt{\bar\alpha_t}x_0}{1-\bar\alpha_t}.$$
Substitute the forward equation $x_t-\sqrt{\bar\alpha_t}x_0=\sqrt{1-\bar\alpha_t}\,\varepsilon$:
$$\nabla_{x_t}\log q(x_t\mid x_0)=-\frac{\sqrt{1-\bar\alpha_t}\,\varepsilon}{1-\bar\alpha_t}=-\frac{\varepsilon}{\sqrt{1-\bar\alpha_t}}.\;\blacksquare$$
Denoising score matching trains $s_\theta$ to match the **conditional** score in expectation; an elementary identity shows this also matches the **marginal** score $\nabla_{x_t}\log p_t(x_t)$. Therefore
$$\boxed{\;s_\theta(x_t,t)=-\,\frac{\varepsilon_\theta(x_t,t)}{\sqrt{1-\bar\alpha_t}}.\;}$$
This is the bridge that lets us plug a DDPM-trained network straight into a score-SDE or probability-flow ODE sampler.

**(b)** The target $\varepsilon\sim\mathcal{N}(0,I)$ has **timestep-independent unit variance**, so the MSE loss has a **uniform scale** over every $t$ and can be trained with a single global learning rate. Predicting $x_0$ directly means the target is at data scale; the residual is multiplied by $\sqrt{\bar\alpha_t}$ which shrinks to $0$ near $t=T$, so the network receives almost no gradient there — it is being asked to recover a clean sample from pure noise. Predicting $\mu_\theta$ has the same problem because $\mu$ scales with $\sqrt{\alpha_t}$. $\varepsilon$-prediction is the unique parameterisation whose target has the same scale at every $t$; this is why Ho et al. (2020) adopted it and why essentially every diffusion model since has followed suit.

**(c)** The DDPM reverse step we derived in Task B5 is a **local-Markov** approximation to the true reverse kernel: it assumes the gap $t\to t-1$ is *small* so that $q(x_{t-1}\mid x_t,x_0)$ is well-approximated by a Gaussian whose mean is linear in $\varepsilon_\theta$. With $T=1000$ each step really is small ($\beta_t\lesssim 0.02$) and the approximation is accurate. If we try to run the *same* updater with only 10 steps on a $T=1000$ schedule, each step now has to jump by 100 indices — the local Gaussian assumption collapses, the posterior at such large gaps is highly non-Gaussian and multi-modal, and the iterates diverge into garbage. This is **exactly** the setting DDIM was designed for: it re-derives a non-Markovian, deterministic update that is **valid for any sub-sequence** of timesteps and therefore lets the same trained $\varepsilon_\theta$ be sampled in, say, 10 or 50 steps without retraining. So the answer to "why 1000" is: 1000 is what the *derivation* of the DDPM step needs in order for its local-Gaussian approximation to be accurate; fewer-step sampling needs a *different* sampler (DDIM, DPM-Solver, consistency distillation, ...), not a shorter run of the same one.
</details>

## Q3 · GANs vs DDPMs — the likelihood-free trade-off  *(8 marks)*

**(a)** *(3 marks)* Neither GANs nor DDPMs evaluate $p_\theta(x)$ directly. Explain, for **each** model, **what** object is learned in place of the likelihood and **how** that object is used to (i) train and (ii) sample. Your answer should make clear that a GAN learns a **sampler** and a **critic**, while a DDPM learns a **noise predictor** that is (up to a constant) an estimate of $\nabla_{x_t}\log p_t(x_t)$.

**(b)** *(3 marks)* Describe the **sampling-cost vs coverage** trade-off. Which model samples in one forward pass, and which needs hundreds of forward passes? Which is prone to **mode collapse**, and **why** does the other one mostly avoid it? Tie the "why" back to the training objective: what is it about the $\varepsilon$-prediction MSE that forces $\varepsilon_\theta$ to be informative at **every** noise level, including the ones near clean data where small errors matter most?

**(c)** *(2 marks)* Given that a DDPM learns $\varepsilon_\theta(x_t,t)$, **why can't we just take $-\varepsilon_\theta$ as a one-shot sampler** — i.e. draw $x_T\sim\mathcal{N}(0,I)$ and return $x_T-\varepsilon_\theta(x_T,T)$ as the sample? Say in one or two sentences what this shortcut is estimating, and why it is fundamentally different from what the iterative reverse chain computes.

<details><summary><b>▸ Answer sketch — Q3</b></summary>

**(a)** **GAN.** Learns two objects: a sampler $G_\theta(z)$ and a critic $D_\phi(x)$. Neither is a density. Training is the adversarial minimax game: $D$ is trained with binary cross-entropy to tell real from fake, and $G$ is trained (via the non-saturating loss) to push its fake outputs toward $D(G(z))=1$. Sampling is trivial — one forward pass $z\to G(z)$ — and the critic $D$ is thrown away at inference time. **DDPM.** Learns a single object: a noise predictor $\varepsilon_\theta(x_t,t)$. By Q2(a) this is equivalent (up to $-1/\sqrt{1-\bar\alpha_t}$) to an estimate of the **score** $\nabla_{x_t}\log p_t(x_t)$ at a whole family of noise levels. Training is the MSE objective $\mathbb E\lVert\varepsilon-\varepsilon_\theta(x_t,t)\rVert^2$, which is denoising score matching. Sampling is iterative: start at $x_T\sim\mathcal{N}(0,I)$, apply the reverse step from Task B5 for $t=T,T-1,\dots,1$. So both models give up on the likelihood but replace it with different auxiliary objects: an **implicit** sampler–critic pair in the GAN case, and an **explicit** score field in the DDPM case.

**(b)** **Cost.** GAN: one forward pass; DDPM: one forward pass **per reverse step**, so $\sim T$ passes ($T\!\approx\!1000$ on images). GANs win sampling cost by 2–3 orders of magnitude. **Coverage.** GANs are famously prone to mode collapse because the generator's loss is fully satisfied by producing *any* output the discriminator accepts as real — nothing in $\mathcal{L}_G^{\text{NS}}$ rewards $G$ for covering all of $p_{\text{data}}$. DDPMs mostly avoid mode collapse: the loss $\mathbb E_{x_0}\lVert\varepsilon-\varepsilon_\theta(x_t,t)\rVert^2$ is taken over **every** training sample $x_0$ and **every** timestep $t$. To drive the loss down the network has to predict the correct noise for $x_t$ values that came from *every* mode at *every* noise level — averaging over training data directly forces coverage. In particular, the tight-scale $\varepsilon$ target near $t\approx 0$ means the model cannot ignore any mode without paying a large loss on samples from that mode.

**(c)** $-\varepsilon_\theta(x_T,T)$ is an estimate of the **score** of $p_T$, which is already $\approx\mathcal{N}(0,I)$ — so the shortcut $x_T-\varepsilon_\theta(x_T,T)$ is essentially a single Langevin half-step against an isotropic Gaussian prior and brings you nowhere near $p_{\text{data}}$. The iterative reverse chain is valuable precisely because it integrates the score $\nabla_{x_t}\log p_t$ along the **whole** path $t:T\to 0$, interleaving many small denoising corrections with stochastic kicks; the one-shot shortcut throws away all the intermediate score estimates $\varepsilon_\theta(x_t,t)$ for $t<T$ — which is where the mass of the learned distribution actually lives. Collapsing 1000 steps to 1 in a principled way is exactly the motivation for consistency models and distillation — it is *not* free.
</details>

---
## Summary

| | **GAN** | **DDPM** |
|---|---|---|
| Family | Implicit / adversarial | Score-based / denoising |
| Learns | Sampler $G_\theta$ + critic $D_\phi$ | Noise predictor $\varepsilon_\theta(x_t,t)$ |
| Loss | Non-saturating BCE: $-\log D(G(z))$ for $G$, $-\log D(x)-\log(1-D(G(z)))$ for $D$ | $\mathbb E\lVert\varepsilon-\varepsilon_\theta(x_t,t)\rVert^2$ (denoising score matching) |
| Samples $x$ via | $x=G(z)$, one forward pass | Reverse chain $x_T\to x_{T-1}\to\cdots\to x_0$, $\sim T$ forward passes |
| Typical failure | **Mode collapse** (generator ignores modes) | Slow sampling (many $\varepsilon_\theta$ evaluations) |
| Fix | WGAN, gradient penalty, non-saturating loss | Faster samplers: DDIM, DPM-Solver, consistency models |

**Two-sentence takeaway.** GANs are the *fastest* way to sample but the *hardest* to train; DDPMs are the *easiest* to train (a single MSE) but the *slowest* to sample. Modern fast diffusion (DDIM, flow matching, consistency models — see the self-study notebook) attempts to keep DDPM's easy training while recovering GAN-like sampling cost.